<a href="https://colab.research.google.com/github/RodriTrooCo9/IA-ProIMAGE/blob/main/Copy_of_IMAGE_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

# Orden importante: primero las bases
pip("peft==0.11.0")
pip("huggingface_hub==0.24.0")
pip("diffusers==0.30.0")
pip("transformers==4.44.0")
pip("accelerate==0.33.0")
pip("safetensors==0.4.3")
pip("gradio==3.50.2")
pip("imageio==2.34.1")
pip("imageio-ffmpeg==0.4.9")
pip("omegaconf")
pip("einops")

subprocess.run(["apt-get", "install", "-qq", "ffmpeg"], capture_output=True)

import torch
print(f"PyTorch : {torch.version}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Dependencias listas")

PyTorch : <module 'torch.version' from '/usr/local/lib/python3.12/dist-packages/torch/version.py'>
CUDA    : False
Dependencias listas


DRIVE + DIRECTORIOS + CONFIGURACION


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, datetime, gc, torch
import numpy as np
from PIL import Image
from pathlib import Path # Implementación de Pathlib para rutas más seguras

# Uso de Pathlib para mejor gestión de rutas
ROOT = Path("/content/drive/MyDrive/AI_Studio_v4")

DIRS = {
    "images"  : ROOT / "images",
    "videos"  : ROOT / "videos",
    "exports" : ROOT / "exports",
    "sessions": ROOT / "sessions",
    "cache"   : ROOT / "model_cache",
    "gallery" : ROOT / "gallery",
}

# Creación de directorios optimizada
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# Configuración de variables de entorno (convertimos Path a string para el OS)
os.environ["HF_HOME"]            = str(DIRS["cache"])
os.environ["TRANSFORMERS_CACHE"] = str(DIRS["cache"])
os.environ["DIFFUSERS_CACHE"]    = str(DIRS["cache"])

LOG = DIRS["sessions"] / "history.jsonl"

def save_log(entry):
    entry["ts"] = datetime.datetime.now().isoformat()
    # Recomendación: Si haces batch, agrupa las entradas antes de escribir
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def load_log(n=20):
    if not LOG.exists():
        return []
    with open(LOG, encoding="utf-8") as f:
        lines = f.readlines()
    return [json.loads(l) for l in lines[-n:]]

def ts():
    return datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# NUEVA FUNCIÓN: Optimización de memoria
def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

# (Los diccionarios STYLES, PRESETS y NEG_DEFAULT se mantienen igual, su estructura es excelente)

print(f"Drive montado en: {ROOT}")
print(f"Dispositivo : {DEVICE.upper()}")

# CONTROL DE ERROR: Verificación segura de la GPU antes de consultar sus propiedades
if DEVICE == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU         : {gpu_name}")
    print(f"VRAM        : {vram_gb:.1f} GB")
else:
    print("VRAM        : N/A (Ejecutando en CPU - El rendimiento será limitado)")

Mounted at /content/drive
Drive montado en: /content/drive/MyDrive/AI_Studio_v4
Dispositivo : CPU
VRAM        : N/A (Ejecutando en CPU - El rendimiento será limitado)


In [6]:
NEG_DEFAULT = "extra digits, fewer digits, lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry"

STYLES = {
    "anime": {
        "trigger": "anime style, masterpiece, best quality, ultra-detailed",
        "positive": "anime style, highly detailed, vibrant colors",
        "negative": "photorealistic, realistic, 3d render"
    },
    "realismo": {
        "trigger": "photorealistic, highly detailed, 8k, raw photo",
        "positive": "professional photography, sharp focus, high dynamic range",
        "negative": "anime, cartoon, graphic, drawing, painting"
    },
    "hiperrealismo": {
        "trigger": "hyperrealistic, intricate detail, unreal engine 5, octane render",
        "positive": "cinematic lighting, extremely detailed, depth of field",
        "negative": "sketch, oil painting, low resolution, simple background"
    }
}

PRESETS = {
    "anime": [
        "1girl, sitting on a rooftop, sunset background, lofi aesthetics",
        "cyberpunk city, neon lights, rainy night, futuristic atmosphere",
        "forest spirit, glowing eyes, mystical woods, ethereal light",
        "magical girl, sparkles, cosmic background, long flowing hair"
    ],
    "realismo": [
        "portrait of an elderly man, wrinkled face, dramatic lighting",
        "mountain landscape, snow-capped peaks, crystal clear lake",
        "street photography, New York City, busy sidewalk, motion blur",
        "macro shot of a bee on a flower, morning dew, bokeh"
    ],
    "hiperrealismo": [
        "abandoned castle, overgrown vines, cinematic lighting, 8k",
        "mechanical eye close-up, complex gears, reflecting a city",
        "underwater coral reef, bioluminescent creatures, high contrast",
        "volcanic eruption, flowing lava, dark smoke, high detail"
    ]
}

print("Configuraciones de Estilos y Presets listas.")

Configuraciones de Estilos y Presets listas.


 PIPELINE DE IMAGENES

In [1]:
import torch, gc, os, cv2
from diffusers import (
    StableDiffusionXLPipeline,
    StableDiffusionXLImg2ImgPipeline,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
    KDPM2AncestralDiscreteScheduler,
)
from PIL import Image
import numpy as np

IMG_MODELS = {
    "RealVisXL 4.0"           : "SG161222/RealVisXL_V4.0",
    "RealVisXL 5.0 Lightning" : "SG161222/RealVisXL_V5.0_Lightning",
    "Juggernaut XL v9"        : "RunDiffusion/Juggernaut-XL-v9",
    "DreamShaper XL Turbo"    : "Lykon/dreamshaper-xl-v2-turbo",
    "SDXL Base"               : "stabilityai/stable-diffusion-xl-base-1.0",
}

SCHEDULERS = {
    "DPM++ 2M Karras" : DPMSolverMultistepScheduler,
    "Euler Ancestral"  : EulerAncestralDiscreteScheduler,
    "KDPM2 Ancestral"  : KDPM2AncestralDiscreteScheduler,
}

pipe_txt2img      = None
pipe_img2img      = None
current_img_model = None

def load_img_model(model_key="RealVisXL 4.0"):
    global pipe_txt2img, pipe_img2img, current_img_model

    if current_img_model == model_key:
        print(f"Modelo ya cargado: {model_key}")
        return

    if pipe_txt2img is not None:
        del pipe_txt2img, pipe_img2img
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    mid = IMG_MODELS[model_key]
    print(f"Cargando {model_key}...")

    kw = dict(
        torch_dtype             = DTYPE,
        use_safetensors         = True,
        safety_checker          = None,
        requires_safety_checker = False,
        add_watermarker         = False,
    )

    pipe_txt2img = StableDiffusionXLPipeline.from_pretrained(
        mid, **kw
    ).to(DEVICE)

    # OPTIMIZACIÓN: Hereda todos los componentes de txt2img automáticamente
    # garantizando 0 consumo extra de VRAM y evitando mapeo manual extenso.
    pipe_img2img = StableDiffusionXLImg2ImgPipeline(
        **pipe_txt2img.components
    )

    for p in [pipe_txt2img, pipe_img2img]:
        p.vae.enable_slicing()
        p.vae.enable_tiling()
        try:
            p.enable_xformers_memory_efficient_attention()
        except:
            p.enable_attention_slicing()

    current_img_model = model_key
    print(f"Modelo listo: {model_key}")


def detect_style(image):
    if isinstance(image, np.ndarray):
        img = Image.fromarray(image)
    else:
        img = image

    # OPTIMIZACIÓN: Redimensionar y convertir la imagen UNA sola vez.
    img_resized = img.convert("RGB").resize((224, 224))
    arr = np.array(img_resized, dtype=np.float32)

    r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]
    max_c   = np.maximum(np.maximum(r, g), b)
    min_c   = np.minimum(np.minimum(r, g), b)

    # Evitar divisiones por cero con 1e-6 de forma segura
    sat   = np.mean(max_c - min_c) / (np.mean(max_c) + 1e-6)
    var_c = np.mean([np.std(r), np.std(g), np.std(b)])

    # Reutilizar el arreglo ya redimensionado para escala de grises
    arr_uint8 = arr.astype(np.uint8)
    gray      = cv2.cvtColor(arr_uint8, cv2.COLOR_RGB2GRAY)
    sharp     = cv2.Laplacian(gray, cv2.CV_64F).var()

    if sat > 0.4 and var_c < 50:
        return "anime"
    elif sharp > 800:
        return "hiperrealismo"
    return "realismo"


def generate_image(prompt, style="auto", negative_prompt=None,
                   steps=30, guidance=7.0, width=1024, height=1024,
                   seed=-1, scheduler_name="DPM++ 2M Karras",
                   num_images=1, input_image=None):

    if style == "auto" and input_image is not None:
        style = detect_style(input_image)
    elif style == "auto":
        style = "realismo"

    style_cfg   = STYLES.get(style, STYLES["realismo"])
    full_prompt = f"{style_cfg['trigger']}, {prompt}" if prompt else style_cfg["positive"]

    if not negative_prompt or not negative_prompt.strip():
        negative_prompt = f"{NEG_DEFAULT}, {style_cfg['negative']}"

    cls = SCHEDULERS[scheduler_name]
    pipe_txt2img.scheduler = cls.from_config(pipe_txt2img.scheduler.config)

    seed_val  = seed if seed != -1 else int(torch.randint(0, 2**32, (1,)).item())
    generator = torch.Generator(DEVICE).manual_seed(seed_val)

    print(f"Generando [{style}] — {full_prompt[:60]}...")

    imgs = pipe_txt2img(
        prompt                = full_prompt,
        negative_prompt       = negative_prompt,
        num_inference_steps   = int(steps),
        guidance_scale        = float(guidance),
        width                 = int(width),
        height                = int(height),
        generator             = generator,
        num_images_per_prompt = int(num_images),
    ).images

    stamp = ts()
    paths = []

    # OPTIMIZACIÓN: I/O de disco
    for i, img in enumerate(imgs):
        p = f"{DIRS['images']}/{style}{stamp}{i}.png"
        g = f"{DIRS['gallery']}/{style}{stamp}{i}.png"
        img.save(p)
        img.save(g)
        paths.append(p)

    save_log({"type": "txt2img", "style": style, "model": current_img_model,
              "prompt": full_prompt, "seed": seed_val, "files": paths})
    print(f"{len(imgs)} imagen(es) guardadas")

    return imgs, paths, seed_val, style


def change_style(init_image, target_style, prompt="",
                 strength=0.65, steps=30, guidance=7.0, seed=-1):

    if isinstance(init_image, np.ndarray):
        init_image = Image.fromarray(init_image)

    init_image   = init_image.convert("RGB")
    source_style = detect_style(init_image)
    style_cfg    = STYLES.get(target_style, STYLES["realismo"])
    full_prompt  = f"{style_cfg['trigger']}, {prompt}" if prompt else style_cfg["positive"]
    neg_prompt   = f"{NEG_DEFAULT}, {style_cfg['negative']}"

    pipe_img2img.scheduler = DPMSolverMultistepScheduler.from_config(
        pipe_img2img.scheduler.config
    )

    seed_val  = seed if seed != -1 else int(torch.randint(0, 2**32, (1,)).item())
    generator = torch.Generator(DEVICE).manual_seed(seed_val)

    print(f"Convirtiendo {source_style} → {target_style}...")

    result = pipe_img2img(
        prompt              = full_prompt,
        negative_prompt     = neg_prompt,
        image               = init_image,
        strength            = float(strength),
        num_inference_steps = int(steps),
        guidance_scale      = float(guidance),
        generator           = generator,
    ).images[0]

    stamp = ts()
    p = f"{DIRS['images']}/{source_style}to{target_style}_{stamp}.png"
    g = f"{DIRS['gallery']}/{source_style}to{target_style}_{stamp}.png"

    result.save(p)
    result.save(g)

    save_log({"type": "style_transfer", "from": source_style,
              "to": target_style, "file": p})
    print(f"Conversion guardada: {p}")

    return result, p


print("Pipeline de imagenes listo")
print("El modelo se cargara al generar la primera imagen")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Pipeline de imagenes listo
El modelo se cargara al generar la primera imagen


PIPELINE DE VIDEO

In [2]:
import torch, gc
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_video, export_to_gif

pipe_video = None

def load_video_model():
    global pipe_video

    if pipe_video is not None:
        print("Pipeline video ya cargado")
        return

    # OPTIMIZACIÓN: Limpieza segura de variables globales del módulo de imágenes
    global pipe_txt2img, pipe_img2img, current_img_model

    if 'pipe_txt2img' in globals() and pipe_txt2img is not None:
        print("Liberando pipeline de imagen para cargar video...")
        del pipe_txt2img

        if 'pipe_img2img' in globals() and pipe_img2img is not None:
            del pipe_img2img

        # Reseteamos el estado del modelo de imagen para evitar conflictos futuros
        if 'current_img_model' in globals():
            current_img_model = None

    # Limpieza profunda de VRAM
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print("Cargando AnimateDiff...")
    adapter = MotionAdapter.from_pretrained(
        "guoyww/animatediff-motion-adapter-v1-5-2",
        torch_dtype=DTYPE,
    )

    pipe_video = AnimateDiffPipeline.from_pretrained(
        "SG161222/Realistic_Vision_V5.1_noVAE",
        motion_adapter          = adapter,
        torch_dtype             = DTYPE,
        safety_checker          = None,
        requires_safety_checker = False,
    ).to(DEVICE)

    pipe_video.scheduler = DDIMScheduler.from_config(
        pipe_video.scheduler.config,
        beta_schedule    = "linear",
        clip_sample      = False,
        timestep_spacing = "linspace",
        steps_offset     = 1,
    )

    pipe_video.vae.enable_slicing()
    # OPTIMIZACIÓN CRÍTICA: Tiling previene OOM al decodificar múltiples frames
    pipe_video.vae.enable_tiling()

    try:
        pipe_video.enable_xformers_memory_efficient_attention()
    except:
        pipe_video.enable_attention_slicing()

    print("AnimateDiff listo")


def generate_video(prompt, style="realismo", steps=25,
                   guidance=7.5, num_frames=16, fps=8, seed=-1):
    load_video_model()

    style_cfg   = STYLES.get(style, STYLES["realismo"])
    full_prompt = f"{style_cfg['trigger']}, {prompt}"
    neg_prompt  = f"{NEG_DEFAULT}, {style_cfg['negative']}"

    seed_val  = int(seed) if seed != -1 else int(torch.randint(0, 2**32, (1,)).item())
    generator = torch.Generator(DEVICE).manual_seed(seed_val)

    # OPTIMIZACIÓN: Forzar liberación de RAM residual justo antes de la carga pesada
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Generando video [{style}] — {num_frames} frames...")

    # Casting estricto en los parámetros para máxima estabilidad
    frames = pipe_video(
        prompt              = full_prompt,
        negative_prompt     = neg_prompt,
        num_inference_steps = int(steps),
        guidance_scale      = float(guidance),
        num_frames          = int(num_frames),
        generator           = generator,
    ).frames[0]

    stamp    = ts()
    mp4_path = f"{DIRS['videos']}/{style}_{stamp}.mp4"
    export_to_video(frames, mp4_path, fps=int(fps))

    save_log({"type": "video", "style": style, "prompt": full_prompt,
              "frames": num_frames, "fps": fps, "file": mp4_path})

    print(f"Video guardado: {mp4_path}")
    return mp4_path

print("Modulo de video listo")

Modulo de video listo


5 — **UPSCALER**

In [3]:
import os, cv2
from PIL import Image
import numpy as np

def upscale_image(image, scale=4):
    # Convertir a numpy array si es PIL Image
    if isinstance(image, Image.Image):
        arr = np.array(image, dtype=np.uint8)
    else:
        arr = image  # Asumimos que ya es un np.ndarray

    h, w = arr.shape[:2]

    # Calcular las nuevas dimensiones una sola vez
    new_w = int(w * scale)
    new_h = int(h * scale)

    # OPTIMIZACIÓN CRÍTICA: cv2.resize procesa la matriz matemática directamente.
    # No es necesario hacer RGB -> BGR -> RGB. Esto ahorra CPU y tiempo.
    upsc = cv2.resize(arr, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)

    result = Image.fromarray(upsc)

    # Generar ruta (el str() asegura compatibilidad si DIRS usa Pathlib)
    stamp = ts()
    p = f"{str(DIRS['exports'])}/upscaled_{stamp}.png"

    # Guardar en disco
    result.save(p)
    save_log({"type": "upscale", "scale": scale, "file": p})

    print(f"Upscaled x{scale} guardado: {p}")
    return result, p

print("Modulo upscaler listo")

Modulo upscaler listo


**6 — INTERFAZ GRADIO MEJORADA**


In [7]:
import gradio as gr
import os

SIZE_LIST  = [512, 640, 768, 896, 1024]
MODEL_LIST = list(IMG_MODELS.keys())
STYLE_LIST = ["auto", "anime", "realismo", "hiperrealismo"]
SCHED_LIST = list(SCHEDULERS.keys())
NEG_DEF    = NEG_DEFAULT

CSS = """
footer { display: none !important; }
.gr-button-primary {
    background: linear-gradient(135deg, #667eea, #764ba2) !important;
    border: none !important;
    color: white !important;
}
.gr-button-secondary {
    border: 2px solid #667eea !important;
    color: #667eea !important;
}
"""

def get_gallery():
    d = str(DIRS["gallery"])
    if not os.path.exists(d):
        return []
    files = [f.path for f in os.scandir(d) if f.name.endswith(".png")]
    files.sort(reverse=True)
    return files[:16]

def load_preset(style, idx):
    p = PRESETS.get(style, PRESETS["realismo"])
    idx = int(idx)
    return p[idx] if idx < len(p) else ""

def ui_generate(model, style, prompt, neg, steps, cfg, w, h, seed, sched, n):
    load_img_model(model)
    imgs, paths, used, detected = generate_image(
        prompt          = prompt,
        style           = style,
        negative_prompt = neg if neg.strip() else None,
        steps           = steps,
        guidance        = cfg,
        width           = w,
        height          = h,
        seed            = seed,
        scheduler_name  = sched,
        num_images      = n,
    )
    info = f"Estilo detectado: {detected}\nSeed: {used}\n" + "\n".join(paths)
    return imgs[0], info, get_gallery()

def ui_change_style(model, image, target, prompt, strength, steps, cfg, seed):
    if image is None:
        return None, "Sube una imagen primero", get_gallery()
    load_img_model(model)
    result, path = change_style(image, target, prompt, strength, steps, cfg, seed)
    return result, f"Guardado: {path}", get_gallery()

def ui_video(style, prompt, steps, cfg, frames, fps, seed):
    path = generate_video(prompt, style, steps, cfg, frames, fps, seed)
    return path, f"Video guardado: {path}"

def ui_upscale(image, scale):
    if image is None:
        return None, "Sube una imagen primero"
    r, p = upscale_image(image, int(scale))
    return r, f"Guardado: {p}"

def ui_history():
    entries = load_log(20)
    if not entries:
        return "Sin historial aun."
    lines = []
    for e in reversed(entries):
        t     = e.get("ts", "?")[:19]
        typ   = e.get("type", "?").upper()
        style = e.get("style", "")
        prom  = e.get("prompt", "")[:60]
        lines.append(f"[{t}] {typ} [{style}]\n  {prom}")
    return "\n\n".join(lines)

with gr.Blocks(title="AI Studio v4", css=CSS, theme=gr.themes.Soft()) as demo:
    gr.Markdown("# AI Studio v4\n### Anime · Realismo · Hiperrealismo")
    with gr.Tabs():
        with gr.Tab("Generar Imagen"):
            with gr.Row():
                with gr.Column(scale=1):
                    g_model = gr.Dropdown(MODEL_LIST, value="RealVisXL 4.0", label="Modelo")
                    g_style = gr.Radio(STYLE_LIST, value="auto", label="Estilo")
                    g_prompt = gr.Textbox(label="Prompt", lines=3)
                    g_neg = gr.Textbox(label="Negative Prompt", lines=2)
                    with gr.Accordion("Presets", open=False):
                        pr_style = gr.Dropdown(["anime", "realismo", "hiperrealismo"], value="anime", label="Estilo")
                        pr_idx = gr.Slider(0, 3, 0, step=1, label="Preset")
                        pr_btn = gr.Button("Cargar")
                        pr_btn.click(load_preset, [pr_style, pr_idx], g_prompt)
                    with gr.Row():
                        g_steps = gr.Slider(5, 60, 30, label="Steps")
                        g_cfg = gr.Slider(1, 15, 7.0, label="CFG")
                    g_btn = gr.Button("Generar", variant="primary")
                with gr.Column(scale=1):
                    g_out = gr.Image(label="Resultado")
                    g_gallery = gr.Gallery(label="Galeria", columns=4)
            g_btn.click(ui_generate, [g_model, g_style, g_prompt, g_neg, g_steps, g_cfg, gr.State(1024), gr.State(1024), gr.State(-1), gr.State("DPM++ 2M Karras"), gr.State(1)], [g_out, gr.State(), g_gallery])

print("Lanzando AI Studio v4...")
demo.queue(default_concurrency_limit=1).launch(share=True, show_error=True)

/tmp/ipykernel_939/2902888595.py:83: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Studio v4", css=CSS, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_939/2902888595.py:83: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="AI Studio v4", css=CSS, theme=gr.themes.Soft()) as demo:


Lanzando AI Studio v4...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fbf94a4f822843908c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
